# Food Image Classification System
**Central Asian University — Image Processing Module**  
**Spring 2025–2026 Resit Examination**

---

**Dataset:** Food-101 (10-class subset)  
**Models:** Custom CNN + MobileNetV2 Transfer Learning  
**Goal:** Classify food images into 10 categories

---

## Table of Contents
1. Setup & Imports
2. Dataset Download & Preparation
3. Exploratory Data Analysis (EDA)
4. Image Preprocessing & Augmentation
5. Model 1: Custom CNN
6. Model 2: Transfer Learning (MobileNetV2)
7. Fine-Tuning
8. Evaluation & Metrics
9. Confusion Matrix
10. Inference on New Images
11. Model Comparison

## 1. Setup & Imports

In [ ]:
# Install dependencies (uncomment if needed)
# !pip install tensorflow opencv-python scikit-learn seaborn matplotlib tqdm -q

import os
import sys
import json
import random
import shutil
import zipfile
import urllib.request
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import cv2

import tensorflow as tf
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, precision_recall_fscore_support
)

# Reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
random.seed(SEED)

# Config
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = 10
CLASSES = [
    'pizza', 'sushi', 'hamburger', 'ice_cream', 'ramen',
    'steak', 'waffles', 'spaghetti_bolognese', 'hot_dog', 'donuts'
]

os.makedirs('dataset/train', exist_ok=True)
os.makedirs('dataset/val', exist_ok=True)
os.makedirs('dataset/test', exist_ok=True)
os.makedirs('results', exist_ok=True)

print(f'TensorFlow version: {tf.__version__}')
print(f'GPU available: {len(tf.config.list_physical_devices("GPU")) > 0}')
print(f'Classes: {CLASSES}')

## 2. Dataset Download & Preparation

**Option A:** Use tf.keras.utils to download Food-101 (recommended).  
**Option B:** Place images manually in `dataset/train/{class_name}/`.

In [ ]:
# Option A: Download using TensorFlow Datasets
# Uncomment to use:
# !pip install tensorflow-datasets -q
# import tensorflow_datasets as tfds
# ds, info = tfds.load('food101', with_info=True, as_supervised=True)

# ─────────────────────────────────────────────────────────────────────────────
# Option B: Synthetic demo dataset (for testing without full download)
# Creates a small demo with random colored images labelled by class
# ─────────────────────────────────────────────────────────────────────────────

def create_demo_dataset(base_path='dataset', n_per_class=50):
    """
    Create a demo dataset with synthetic images.
    In real use, replace with actual Food-101 images.
    """
    splits = {'train': 35, 'val': 8, 'test': 7}
    color_map = {
        'pizza': (220, 100, 50), 'sushi': (180, 220, 180),
        'hamburger': (180, 120, 60), 'ice_cream': (255, 230, 200),
        'ramen': (240, 200, 100), 'steak': (150, 60, 40),
        'waffles': (230, 190, 120), 'spaghetti_bolognese': (200, 80, 50),
        'hot_dog': (240, 150, 80), 'donuts': (220, 160, 100)
    }

    for split, count in splits.items():
        for cls in CLASSES:
            cls_path = os.path.join(base_path, split, cls)
            os.makedirs(cls_path, exist_ok=True)
            color = color_map[cls]
            for i in range(count):
                img = np.ones((224, 224, 3), dtype=np.uint8)
                img[:, :] = color
                # Add some noise to distinguish images
                noise = np.random.randint(0, 40, (224, 224, 3), dtype=np.uint8)
                img = np.clip(img.astype(int) + noise - 20, 0, 255).astype(np.uint8)
                # Add geometric shape for visual variety
                cx, cy = np.random.randint(50, 174, 2)
                r = np.random.randint(20, 60)
                cv2.circle(img, (cx, cy), r, (255, 255, 255), -1)
                cv2.imwrite(os.path.join(cls_path, f'{cls}_{i:03d}.jpg'), img)

    print('Demo dataset created!')
    for split in splits:
        total = sum(len(os.listdir(os.path.join(base_path, split, c))) for c in CLASSES)
        print(f'  {split}: {total} images')

# Run demo (comment out when using real Food-101 data)
create_demo_dataset()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Count images per class per split
data_summary = []
for split in ['train', 'val', 'test']:
    for cls in CLASSES:
        path = f'dataset/{split}/{cls}'
        count = len(os.listdir(path)) if os.path.exists(path) else 0
        data_summary.append({'split': split, 'class': cls, 'count': count})

df = pd.DataFrame(data_summary)
pivot = df.pivot(index='class', columns='split', values='count')
pivot['total'] = pivot.sum(axis=1)
print('Dataset Summary:')
print(pivot.to_string())
print(f'\nTotal images: {pivot["total"].sum()}')

In [ ]:
# Class distribution bar chart
train_counts = df[df['split'] == 'train'].set_index('class')['count']

fig, ax = plt.subplots(figsize=(13, 5))
colors = plt.cm.Set2(np.linspace(0, 1, NUM_CLASSES))
bars = ax.bar(train_counts.index, train_counts.values, color=colors, edgecolor='white', linewidth=0.8)

for bar, count in zip(bars, train_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            str(count), ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_title('Training Set — Class Distribution', fontsize=15, fontweight='bold', pad=15)
ax.set_xlabel('Food Category', fontsize=12)
ax.set_ylabel('Number of Images', fontsize=12)
ax.tick_params(axis='x', rotation=40)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('results/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/class_distribution.png')

In [ ]:
# Display sample images in grid
fig, axes = plt.subplots(2, 5, figsize=(16, 7))
axes = axes.flatten()

for i, cls in enumerate(CLASSES):
    cls_path = f'dataset/train/{cls}'
    imgs = os.listdir(cls_path)
    if imgs:
        img = cv2.imread(os.path.join(cls_path, imgs[0]))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        axes[i].imshow(img)
    axes[i].set_title(cls.replace('_', '\n').title(), fontsize=9, fontweight='bold')
    axes[i].axis('off')

plt.suptitle('Sample Image per Class', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('results/sample_images.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Image Preprocessing & Augmentation

In [ ]:
# Data Generators
train_datagen = ImageDataGenerator(
    rescale=1.0/255.0,
    rotation_range=20,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.1,
    zoom_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest'
)

val_test_datagen = ImageDataGenerator(rescale=1.0/255.0)

train_gen = train_datagen.flow_from_directory(
    'dataset/train', target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', shuffle=True, seed=SEED
)
val_gen = val_test_datagen.flow_from_directory(
    'dataset/val', target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', shuffle=False
)
test_gen = val_test_datagen.flow_from_directory(
    'dataset/test', target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', shuffle=False
)

print(f'Class mapping: {train_gen.class_indices}')
print(f'Train steps per epoch: {len(train_gen)}')
print(f'Input shape: {next(iter(train_gen))[0].shape}')

In [ ]:
# Visualize augmentation effects on one image
sample_class = CLASSES[0]
sample_img_path = os.path.join('dataset/train', sample_class,
                                os.listdir(f'dataset/train/{sample_class}')[0])
original = cv2.imread(sample_img_path)
original = cv2.cvtColor(original, cv2.COLOR_BGR2RGB)
original = cv2.resize(original, IMG_SIZE)

aug_datagen = ImageDataGenerator(
    rotation_range=30, width_shift_range=0.2,
    height_shift_range=0.2, zoom_range=0.3,
    horizontal_flip=True, brightness_range=[0.5, 1.5]
)
aug_gen = aug_datagen.flow(np.expand_dims(original, 0), batch_size=1)

fig, axes = plt.subplots(2, 5, figsize=(16, 7))
axes = axes.flatten()
axes[0].imshow(original)
axes[0].set_title('Original', fontweight='bold', fontsize=10)
axes[0].axis('off')

aug_labels = ['Rotation', 'H-Flip', 'Zoom', 'Brightness', 'Shift',
              'Shear', 'Combined', 'Aug 8', 'Aug 9']
for i in range(1, 10):
    aug_img = next(aug_gen)[0].astype(np.uint8)
    axes[i].imshow(np.clip(aug_img, 0, 255))
    axes[i].set_title(aug_labels[i-1] if i-1 < len(aug_labels) else f'Aug {i}',
                      fontsize=9)
    axes[i].axis('off')

plt.suptitle('Data Augmentation Examples', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('results/augmentation_examples.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/augmentation_examples.png')

## 5. Model 1 — Custom CNN

In [ ]:
def build_custom_cnn(num_classes=NUM_CLASSES):
    """Custom CNN with 4 convolutional blocks."""
    model = models.Sequential(name='Custom_CNN', layers=[
        # Block 1: 224x224 -> 112x112
        layers.Conv2D(32, (3,3), activation='relu', padding='same',
                      input_shape=(*IMG_SIZE, 3)),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2,2)),

        # Block 2: 112x112 -> 56x56
        layers.Conv2D(64, (3,3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2,2)),

        # Block 3: 56x56 -> 28x28
        layers.Conv2D(128, (3,3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2,2)),

        # Block 4: 28x28 -> 14x14
        layers.Conv2D(256, (3,3), activation='relu', padding='same',
                      kernel_regularizer=regularizers.l2(1e-4)),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2,2)),

        # Classifier
        layers.GlobalAveragePooling2D(),
        layers.Dense(512, activation='relu',
                     kernel_regularizer=regularizers.l2(1e-4)),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax')
    ])

    model.compile(
        optimizer=Adam(learning_rate=1e-3),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

cnn_model = build_custom_cnn()
cnn_model.summary()

In [ ]:
callbacks_cnn = [
    EarlyStopping(monitor='val_accuracy', patience=7,
                  restore_best_weights=True, verbose=1),
    ModelCheckpoint('results/cnn_best.h5', monitor='val_accuracy',
                    save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3,
                      min_lr=1e-7, verbose=1)
]

print('Training Custom CNN...')
history_cnn = cnn_model.fit(
    train_gen,
    epochs=30,
    validation_data=val_gen,
    callbacks=callbacks_cnn,
    verbose=1
)

## 6. Model 2 — Transfer Learning (MobileNetV2)

In [ ]:
# Build MobileNetV2 model
base_model = MobileNetV2(input_shape=(*IMG_SIZE, 3),
                          include_top=False, weights='imagenet')
base_model.trainable = False  # Freeze base for feature extraction

inputs = tf.keras.Input(shape=(*IMG_SIZE, 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.4)(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

mobilenet_model = tf.keras.Model(inputs, outputs, name='MobileNetV2_Classifier')
mobilenet_model.compile(
    optimizer=Adam(1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

trainable_params = sum(tf.keras.backend.count_params(w)
                       for w in mobilenet_model.trainable_weights)
total_params = mobilenet_model.count_params()
print(f'Total parameters:     {total_params:,}')
print(f'Trainable parameters: {trainable_params:,} (new head only)')

In [ ]:
# Phase 1: Feature extraction
callbacks_mn = [
    EarlyStopping(monitor='val_accuracy', patience=5,
                  restore_best_weights=True, verbose=1),
    ModelCheckpoint('results/mobilenet_best.h5', monitor='val_accuracy',
                    save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1)
]

print('Phase 1: Feature extraction (base model frozen)...')
history_p1 = mobilenet_model.fit(
    train_gen, epochs=15, validation_data=val_gen,
    callbacks=callbacks_mn, verbose=1
)

print(f'Phase 1 best val accuracy: {max(history_p1.history["val_accuracy"]):.4f}')

## 7. Fine-Tuning

In [ ]:
# Unfreeze top layers for fine-tuning
FINE_TUNE_AT = 100  # Unfreeze from layer 100 onwards
base_model.trainable = True

for layer in base_model.layers[:FINE_TUNE_AT]:
    layer.trainable = False

trainable_now = sum(1 for l in base_model.layers if l.trainable)
print(f'Unfrozen base layers: {trainable_now} (from layer {FINE_TUNE_AT})')

mobilenet_model.compile(
    optimizer=Adam(learning_rate=1e-5),  # Very low LR for fine-tuning
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print('\nPhase 2: Fine-tuning (top layers unfrozen)...')
history_p2 = mobilenet_model.fit(
    train_gen, epochs=10, validation_data=val_gen,
    callbacks=callbacks_mn, verbose=1
)

# Merge histories for plotting
history_mobilenet = {
    k: history_p1.history[k] + history_p2.history[k]
    for k in history_p1.history
}

## 8. Training Curves

In [ ]:
def plot_history(history, title='Model', save_name='training'):
    """Plot accuracy and loss curves side by side."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    epochs = range(1, len(history['accuracy']) + 1)
    phase_end = len(history_p1.history['accuracy'])  # For MobileNet phases

    # Accuracy
    ax1.plot(epochs, history['accuracy'], 'b-', label='Train', linewidth=2)
    ax1.plot(epochs, history['val_accuracy'], 'r--', label='Validation', linewidth=2)
    if title == 'MobileNetV2':
        ax1.axvline(x=phase_end, color='gray', linestyle=':', label='Fine-tune start')
    ax1.set_title(f'{title} — Accuracy', fontsize=13, fontweight='bold')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Accuracy')
    ax1.legend(); ax1.grid(alpha=0.3)
    ax1.set_ylim(0, 1)

    # Loss
    ax2.plot(epochs, history['loss'], 'b-', label='Train', linewidth=2)
    ax2.plot(epochs, history['val_loss'], 'r--', label='Validation', linewidth=2)
    if title == 'MobileNetV2':
        ax2.axvline(x=phase_end, color='gray', linestyle=':', label='Fine-tune start')
    ax2.set_title(f'{title} — Loss', fontsize=13, fontweight='bold')
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss')
    ax2.legend(); ax2.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(f'results/{save_name}_curves.png', dpi=150, bbox_inches='tight')
    plt.show()

plot_history(history_cnn.history, 'Custom CNN', 'cnn')
plot_history(history_mobilenet, 'MobileNetV2', 'mobilenet')

## 9. Evaluation & Confusion Matrix

In [ ]:
def full_evaluation(model, test_gen, model_name='Model'):
    """Complete evaluation: metrics + confusion matrix."""
    print(f'\nEvaluating {model_name}...')

    y_pred_probs = model.predict(test_gen, verbose=0)
    y_pred = np.argmax(y_pred_probs, axis=1)
    y_true = test_gen.classes
    class_names = list(test_gen.class_indices.keys())

    acc = accuracy_score(y_true, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted')

    print(f'{'='*45}')
    print(f'  {model_name} — Test Results')
    print(f'{'='*45}')
    print(f'  Accuracy:  {acc:.4f} ({acc*100:.2f}%)')
    print(f'  Precision: {prec:.4f}')
    print(f'  Recall:    {rec:.4f}')
    print(f'  F1-Score:  {f1:.4f}')
    print(f'{'='*45}')
    print(classification_report(y_true, y_pred, target_names=class_names))

    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)

    fig, ax = plt.subplots(figsize=(11, 9))
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names,
                ax=ax, linewidths=0.5)
    ax.set_title(f'{model_name} — Confusion Matrix (Normalized)',
                 fontsize=14, fontweight='bold')
    ax.set_ylabel('True Label', fontsize=11)
    ax.set_xlabel('Predicted Label', fontsize=11)
    ax.tick_params(axis='x', rotation=40)
    plt.tight_layout()
    save_name = model_name.lower().replace(' ', '_')
    plt.savefig(f'results/{save_name}_confusion_matrix.png', dpi=150, bbox_inches='tight')
    plt.show()

    return {'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1}

metrics_cnn = full_evaluation(cnn_model, test_gen, 'Custom CNN')
metrics_mn = full_evaluation(mobilenet_model, test_gen, 'MobileNetV2')

## 10. Inference on New Images

In [ ]:
def predict_single(image_path, model, class_names=CLASSES, top_k=3):
    """Predict food class from a single image."""
    img = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(img_rgb, IMG_SIZE)
    img_normalized = img_resized.astype('float32') / 255.0
    img_batch = np.expand_dims(img_normalized, axis=0)

    probs = model.predict(img_batch, verbose=0)[0]
    top_k_idx = np.argsort(probs)[::-1][:top_k]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    ax1.imshow(img_rgb)
    ax1.set_title('Input Image', fontsize=13, fontweight='bold')
    ax1.axis('off')

    names = [class_names[i].replace('_', ' ').title() for i in top_k_idx]
    confs = [probs[i] for i in top_k_idx]
    colors = ['#E74C3C'] + ['#3498DB'] * (top_k - 1)

    ax2.barh(names[::-1], confs[::-1], color=colors[::-1], edgecolor='white')
    for i, (n, c) in enumerate(zip(names[::-1], confs[::-1])):
        ax2.text(c + 0.01, i, f'{c*100:.1f}%', va='center', fontsize=11, fontweight='bold')
    ax2.set_xlim(0, 1.15)
    ax2.set_title(f'Top-{top_k} Predictions', fontsize=13, fontweight='bold')
    ax2.set_xlabel('Confidence')
    ax2.grid(axis='x', alpha=0.3)

    plt.suptitle(f'Prediction: {names[0]} ({confs[0]*100:.1f}%)',
                 fontsize=14, fontweight='bold', color='#C0392B')
    plt.tight_layout()
    plt.savefig('results/sample_prediction.png', dpi=150, bbox_inches='tight')
    plt.show()

    return names[0], confs[0]

# Test on a sample from the test set
sample_class = CLASSES[0]
sample_imgs = os.listdir(f'dataset/test/{sample_class}')
if sample_imgs:
    sample_path = f'dataset/test/{sample_class}/{sample_imgs[0]}'
    name, conf = predict_single(sample_path, mobilenet_model)
    print(f'Predicted: {name} ({conf*100:.1f}%)')

## 11. Model Comparison

In [ ]:
# Side-by-side model comparison
comparison = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score'],
    'Custom CNN': [
        metrics_cnn['accuracy'], metrics_cnn['precision'],
        metrics_cnn['recall'], metrics_cnn['f1']
    ],
    'MobileNetV2': [
        metrics_mn['accuracy'], metrics_mn['precision'],
        metrics_mn['recall'], metrics_mn['f1']
    ]
})

print('\nModel Comparison Table:')
print(comparison.to_string(index=False, float_format='{:.4f}'.format))

# Bar chart comparison
x = np.arange(len(comparison['Metric']))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
b1 = ax.bar(x - width/2, comparison['Custom CNN'], width,
             label='Custom CNN', color='steelblue', edgecolor='white')
b2 = ax.bar(x + width/2, comparison['MobileNetV2'], width,
             label='MobileNetV2', color='tomato', edgecolor='white')

for bar in list(b1) + list(b2):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(comparison['Metric'])
ax.set_ylim(0, 1.15)
ax.set_title('Custom CNN vs MobileNetV2 — Performance Comparison',
              fontsize=13, fontweight='bold')
ax.set_ylabel('Score')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('results/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/model_comparison.png')

In [ ]:
print('\n' + '='*55)
print(' Project Complete!')
print('='*55)
print(f' Best model: MobileNetV2')
print(f' Test Accuracy: {metrics_mn["accuracy"]*100:.2f}%')
print(f' F1-Score: {metrics_mn["f1"]:.4f}')
print('='*55)
print('\nFiles saved to results/:')
for f in sorted(os.listdir('results/')):
    print(f'  • {f}')